# Segmentation Benchmark Results

This notebook reads both legacy `summary.jsonl` and new `benchmark_summary.jsonl`, then separates full holdout runs from fast validation-only runs.


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / "neurosned").exists():
            return path
        nested = path / "neurosned"
        if (nested / "data").exists() and (nested / "neurosned").exists():
            return nested
    raise FileNotFoundError("Could not find neurosned project root.")


PROJECT_ROOT = find_project_root()
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR


PosixPath('/home/qdata/ayana_experiments/neurosned/artifacts')

## Load And Normalize Summaries


In [2]:
records = []
for path in [ARTIFACTS_DIR / "summary.jsonl", ARTIFACTS_DIR / "benchmark_summary.jsonl"]:
    if path.exists():
        with path.open() as f:
            for line in f:
                if line.strip():
                    row = json.loads(line)
                    row["summary_source"] = path.name
                    records.append(row)

summary = pd.DataFrame(records)
if summary.empty:
    raise FileNotFoundError("No summary rows found in artifacts.")

if "experiment" not in summary:
    summary["experiment"] = pd.NA
summary["experiment"] = summary["experiment"].fillna(summary.get("model"))
if "experiment_group" not in summary:
    summary["experiment_group"] = pd.NA
summary["experiment_group"] = summary["experiment_group"].fillna(summary["experiment"])

for col in ["seed", "train_temperature", "eval_temperature"]:
    if col not in summary:
        summary[col] = pd.NA
if "temperature" in summary:
    summary["train_temperature"] = summary["train_temperature"].fillna(summary["temperature"])
    summary["eval_temperature"] = summary["eval_temperature"].fillna(summary["temperature"])

summary["is_complete"] = summary["best_val_nrmse"].notna()
summary["has_holdout"] = summary["holdout_nrmse"].notna()
summary["run_kind"] = summary["has_holdout"].map({True: "full/holdout", False: "validation_only"})
summary.loc[~summary["is_complete"], "run_kind"] = "incomplete"

summary.shape


(40, 36)

## Full Runs With Holdout

Use this table for final model comparison. These rows have both validation and holdout metrics.


In [3]:
holdout_cols = [
    "experiment_group",
    "experiment",
    "model",
    "best_epoch",
    "best_val_nrmse",
    "holdout_nrmse",
    "batch_size",
    "lr",
    "sigma",
    "lambda_time",
    "lambda_ce",
    "train_temperature",
    "eval_temperature",
    "status",
    "run_dir",
]

holdout_results = summary[summary["has_holdout"]].copy()
holdout_results = holdout_results.sort_values(["holdout_nrmse", "best_val_nrmse"], na_position="last")
holdout_results[holdout_cols].reset_index(drop=True)


,experiment_group,experiment,model,best_epoch,best_val_nrmse,holdout_nrmse,batch_size,lr,sigma,lambda_time,lambda_ce,train_temperature,eval_temperature,status,run_dir
0,SneddySegUNet1D_ce05_lt2,SneddySegUNet1D_ce05_lt2_seed42,SneddySegUNet1D,9.0,923.085503,930.830876,512,0.001,0.15,2.0,0.5,0.65,0.65,holdout_evaluated,artifacts/SneddySegUNet1D_ce05_lt2_seed42__Sne...
1,SneddySegUNet1D_ce05_lt2_full,SneddySegUNet1D_ce05_lt2_full,SneddySegUNet1D,9.0,922.890410,930.904217,512,0.001,0.15,2.0,0.5,0.65,0.65,holdout_evaluated,artifacts/SneddySegUNet1D_ce05_lt2_full__Snedd...
2,SneddySegUNet1D_ce05_lt2,SneddySegUNet1D_ce05_lt2_seed42,SneddySegUNet1D,13.0,922.883478,934.058796,512,0.001,0.15,2.0,0.5,0.65,0.65,holdout_evaluated,artifacts/SneddySegUNet1D_ce05_lt2_seed42__Sne...
3,SneddySegUNet1D_sig012,SneddySegUNet1D_sig012_seed42,SneddySegUNet1D,13.0,922.582131,934.493787,512,0.001,0.12,3.0,0.0,0.65,0.65,holdout_evaluated,artifacts/SneddySegUNet1D_sig012_seed42__Snedd...
4,SneddySegUNet1D_sig012,SneddySegUNet1D_sig012_seed42,SneddySegUNet1D,13.0,922.594492,934.503950,512,0.001,0.12,3.0,0.0,0.65,0.65,holdout_evaluated,artifacts/SneddySegUNet1D_sig012_seed42__Snedd...
5,SneddySegUNet1D_sig012_full,SneddySegUNet1D_sig012_full,SneddySegUNet1D,13.0,922.596647,934.506638,512,0.001,0.12,3.0,0.0,0.65,0.65,holdout_evaluated,artifacts/SneddySegUNet1D_sig012_full__SneddyS...
6,SneddySegUNet1D_baseline,SneddySegUNet1D_baseline_seed42,SneddySegUNet1D,15.0,924.977245,934.515103,512,0.001,0.15,3.0,0.0,0.65,0.65,holdout_evaluated,artifacts/SneddySegUNet1D_baseline_seed42__Sne...
7,SneddySegUNet1D_baseline,SneddySegUNet1D_baseline_seed42,SneddySegUNet1D,15.0,925.023027,934.531640,512,0.001,0.15,3.0,0.0,0.65,0.65,holdout_evaluated,artifacts/SneddySegUNet1D_baseline_seed42__Sne...
8,SneddySegUNet1D,SneddySegUNet1D,SneddySegUNet1D,14.0,922.368800,934.653829,512,0.001,0.15,3.0,0.0,0.65,0.65,holdout_evaluated,artifacts/SneddySegUNet1D__c96_w2_d5__bs512__l...
9,SneddySegUNet1D_sig012,SneddySegUNet1D_sig012_seed43,SneddySegUNet1D,10.0,920.960674,937.038525,512,0.001,0.12,3.0,0.0,0.65,0.65,holdout_evaluated,artifacts/SneddySegUNet1D_sig012_seed43__Snedd...


## Fast Validation-Only Runs

These are useful for parameter search. `holdout_nrmse` is intentionally empty for these rows.


In [4]:
validation_cols = [
    "experiment_group",
    "experiment",
    "model",
    "best_epoch",
    "best_val_nrmse",
    "batch_size",
    "lr",
    "sigma",
    "lambda_time",
    "lambda_ce",
    "train_temperature",
    "eval_temperature",
    "status",
    "run_dir",
]

validation_results = summary[(summary["is_complete"]) & (~summary["has_holdout"])].copy()
validation_results = validation_results.sort_values("best_val_nrmse", na_position="last")
validation_results[validation_cols].reset_index(drop=True)


,experiment_group,experiment,model,best_epoch,best_val_nrmse,batch_size,lr,sigma,lambda_time,lambda_ce,train_temperature,eval_temperature,status,run_dir
0,SneddySegUNet1D_sig012_fast,SneddySegUNet1D_sig012_fast,SneddySegUNet1D,13.0,922.582791,512,0.001000,0.12,3.0,0.0,0.65,0.65,training_finished,artifacts/SneddySegUNet1D_sig012_fast__SneddyS...
1,SneddySegUNet1D_ce05_lt2_fast,SneddySegUNet1D_ce05_lt2_fast,SneddySegUNet1D,9.0,923.080052,512,0.001000,0.15,2.0,0.5,0.65,0.65,training_finished,artifacts/SneddySegUNet1D_ce05_lt2_fast__Snedd...
2,SneddySegUNet1D_bs256_fast,SneddySegUNet1D_bs256_fast,SneddySegUNet1D,16.0,924.129589,256,0.001000,0.15,3.0,0.0,0.65,0.65,training_finished,artifacts/SneddySegUNet1D_bs256_fast__SneddySe...
3,SneddySegUNet1D_tau075_fast,SneddySegUNet1D_tau075_fast,SneddySegUNet1D,9.0,924.883953,512,0.001000,0.15,3.0,0.0,0.75,0.75,training_finished,artifacts/SneddySegUNet1D_tau075_fast__SneddyS...
4,SneddySegUNet1D_ce02_lt3_fast,SneddySegUNet1D_ce02_lt3_fast,SneddySegUNet1D,15.0,924.940580,512,0.001000,0.15,3.0,0.2,0.65,0.65,training_finished,artifacts/SneddySegUNet1D_ce02_lt3_fast__Snedd...
5,SneddySegUNet1D_baseline_fast,SneddySegUNet1D_baseline_fast,SneddySegUNet1D,15.0,925.033323,512,0.001000,0.15,3.0,0.0,0.65,0.65,training_finished,artifacts/SneddySegUNet1D_baseline_fast__Snedd...
6,SneddySegUNet1D_sig018_fast,SneddySegUNet1D_sig018_fast,SneddySegUNet1D,13.0,926.096747,512,0.001000,0.18,3.0,0.0,0.65,0.65,training_finished,artifacts/SneddySegUNet1D_sig018_fast__SneddyS...
7,SneddySegUNet1D_lr7e4_fast,SneddySegUNet1D_lr7e4_fast,SneddySegUNet1D,9.0,928.540414,512,0.000700,0.15,3.0,0.0,0.65,0.65,training_finished,artifacts/SneddySegUNet1D_lr7e4_fast__SneddySe...
8,SneddySegUNet1D_sig012,SneddySegUNet1D_sig012_seed44,SneddySegUNet1D,10.0,930.122221,512,0.001000,0.12,3.0,0.0,0.65,0.65,training_best_updated,artifacts/SneddySegUNet1D_sig012_seed44__Snedd...
9,SneddySegUNet1D_notebook_parity_scratch_full,SneddySegUNet1D_notebook_parity_scratch_full,SneddySegUNet1D,100.0,932.665879,2000,0.000063,0.15,3.0,1.0,0.65,0.65,training_finished,artifacts/SneddySegUNet1D_notebook_parity_scra...


## Incomplete Runs

These rows usually come from interrupted notebook cells. They are kept for traceability but should not be compared.


In [5]:
incomplete_cols = ["created_at", "experiment", "model", "status", "run_dir"]
incomplete = summary[~summary["is_complete"]].copy()
incomplete[incomplete_cols].sort_values("created_at", ascending=False).reset_index(drop=True)


,created_at,experiment,model,status,run_dir
0,2026-06-19T19:17:47Z,SneddySegUNet1D_notebook_parity_scratch_full,SneddySegUNet1D,training_started,artifacts/SneddySegUNet1D_notebook_parity_scra...
1,2026-06-17T12:13:27Z,SneddySegUNet1D_baseline,SneddySegUNet1D,training_started,artifacts/SneddySegUNet1D_baseline__SneddySegU...
2,2026-06-17T11:52:46Z,SneddySegUNet1D_baseline,SneddySegUNet1D,training_started,artifacts/SneddySegUNet1D_baseline__SneddySegU...


## Grouped Summary


In [6]:
summary[summary["is_complete"]].groupby(["experiment_group", "model", "run_kind"], dropna=False)[["best_val_nrmse", "holdout_nrmse"]].agg(["count", "mean", "min", "std"])


best_val_nrmse  \
                                                                                              count   
experiment_group                             model                   run_kind                         
AttentionSneddyUnet                          AttentionSneddyUnet     full/holdout                 1   
EEGInceptionSeg1D                            EEGInceptionSeg1D       full/holdout                 1   
FactorizationSneddyUnet                      FactorizationSneddyUnet full/holdout                 1   
RecurrentSneddyUnet                          RecurrentSneddyUnet     full/holdout                 1   
SneddySegUNet1D                              SneddySegUNet1D         full/holdout                 1   
SneddySegUNet1D_baseline                     SneddySegUNet1D         full/holdout                 6   
SneddySegUNet1D_baseline_fast                SneddySegUNet1D         validation_only              1   
SneddySegUNet1D_bs256_fast                   SneddySegUNet1D         validation_only              1   
SneddySegUNet1D_ce02_lt3_fast                SneddySegUNet1D         validation_only              1   
SneddySegUNet1D_ce05_lt2                     SneddySegUNet1D         full/holdout                 6   
SneddySegUNet1D_ce05_lt2_fast                SneddySegUNet1D         validation_only              1   
SneddySegUNet1D_ce05_lt2_full                SneddySegUNet1D         full/holdout                 1   
SneddySegUNet1D_lr3e4_fast                   SneddySegUNet1D         validation_only              1   
SneddySegUNet1D_lr7e4_fast                   SneddySegUNet1D         validation_only              1   
SneddySegUNet1D_notebook_parity_scratch_demo SneddySegUNet1D         validation_only              1   
SneddySegUNet1D_notebook_parity_scratch_full SneddySegUNet1D         validation_only              2   
SneddySegUNet1D_sig012                       SneddySegUNet1D         full/holdout                 5   
                                                                     validation_only              1   
SneddySegUNet1D_sig012_fast                  SneddySegUNet1D         validation_only              1   
SneddySegUNet1D_sig012_full                  SneddySegUNet1D         full/holdout                 1   
SneddySegUNet1D_sig018_fast                  SneddySegUNet1D         validation_only              1   
SneddySegUNet1D_tau075_fast                  SneddySegUNet1D         validation_only              1   

                                                                                                  \
                                                                                            mean   
experiment_group                             model                   run_kind                      
AttentionSneddyUnet                          AttentionSneddyUnet     full/holdout     925.130914   
EEGInceptionSeg1D                            EEGInceptionSeg1D       full/holdout     932.703288   
FactorizationSneddyUnet                      FactorizationSneddyUnet full/holdout     922.841575   
RecurrentSneddyUnet                          RecurrentSneddyUnet     full/holdout     929.449144   
SneddySegUNet1D                              SneddySegUNet1D         full/holdout     922.368800   
SneddySegUNet1D_baseline                     SneddySegUNet1D         full/holdout     926.012322   
SneddySegUNet1D_baseline_fast                SneddySegUNet1D         validation_only  925.033323   
SneddySegUNet1D_bs256_fast                   SneddySegUNet1D         validation_only  924.129589   
SneddySegUNet1D_ce02_lt3_fast                SneddySegUNet1D         validation_only  924.940580   
SneddySegUNet1D_ce05_lt2                     SneddySegUNet1D         full/holdout     923.909475   
SneddySegUNet1D_ce05_lt2_fast                SneddySegUNet1D         validation_only  923.080052   
SneddySegUNet1D_ce05_lt2_full                SneddySegUNet1D         full/holdout     922.890410   
SneddySeg